# Correction — Exercice 4 ★★★☆☆ — Refactoring, et un piège à connaître

> À ouvrir seulement en cas de blocage ! L'exercice est dans `exercice4.ipynb`.

Lancez les cellules dans l'ordre (**Run All**) : les tests doivent tous être ✅.

## Énoncé

Voici l'ancienne version, dupliquée entre SQLAlchemy et Pydantic :

```python
class Product(Base):
    __tablename__ = "products"
    id = Column(Integer, primary_key=True)
    name = Column(String)

class ProductCreate(BaseModel):
    name: str
```

Créez `Product` (la vraie table, avec `id`, `name`, `price: float`) **et** `ProductCreate` (un schéma d'entrée, sans table, avec `name` et `price`). Sur `ProductCreate` uniquement, `price` doit être **strictement positif**.

Essayez d'abord d'instancier `Product(name="x", price=-10)` puis `ProductCreate(name="x", price=-10)`. Comparez les résultats : c'est le piège de cet exercice.

In [ ]:
from sqlmodel import SQLModel, Field, Relationship
from sqlalchemy.orm import clear_mappers

## Correction

In [ ]:
# Ne pas toucher : permet de relancer cette cellule sans erreur.
SQLModel.metadata.clear()
clear_mappers()

class Product(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    price: float


# Le piège : une classe table=True n'est PAS validée à la création,
# donc Product(price=-10) passe. La contrainte gt=0 va sur le schéma d'entrée.
class ProductCreate(SQLModel):
    name: str
    price: float = Field(gt=0)


### Explication

**Le piège :** une classe `table=True` n'est **pas validée** quand on crée l'objet.

```python
Product(name="x", price=-10)        # aucune erreur
ProductCreate(name="x", price=-10)  # ValidationError
```

`gt` veut dire *greater than* (strictement supérieur). La contrainte se met donc sur le schéma d'entrée `ProductCreate`, qui valide les données de l'API **avant** de construire l'objet de table. La cellule « Essai » ci-dessous le montre.

In [ ]:
# Essai : comparez les deux résultats
print("Product(price=-10)       ->", Product(name="x", price=-10))
try:
    ProductCreate(name="x", price=-10)
except Exception as erreur:
    print("ProductCreate(price=-10) -> erreur :", type(erreur).__name__)


## Vérification

In [ ]:
import pytest

def get_fields(model):
    """Compatible pydantic v1 (SQLModel ancien) et v2 (SQLModel récent)."""
    if hasattr(model, "model_fields"):
        return model.model_fields
    return model.__fields__


def is_table(model) -> bool:
    return getattr(model, "__table__", None) is not None


# ---------------------------------------------------------------------------
# Exercice 4 — Product (refactoring + contrainte de validation)
# ---------------------------------------------------------------------------

def test_exercice4_product_existe():
    assert hasattr(ex, "Product"), "La classe Product est introuvable."


def test_exercice4_product_est_une_table():
    assert is_table(ex.Product), "Product doit être déclarée avec table=True."


def test_exercice4_product_a_les_bons_champs():
    fields = get_fields(ex.Product)
    for champ in ("id", "name", "price"):
        assert champ in fields, f"Le champ '{champ}' est manquant sur Product."


def test_exercice4_product_instanciation_valide():
    p = ex.Product(name="Clavier mécanique", price=49.90)
    assert p.name == "Clavier mécanique"
    assert p.price == 49.90


def test_exercice4_productcreate_existe():
    assert hasattr(ex, "ProductCreate"), "La classe ProductCreate est introuvable."


def test_exercice4_productcreate_pas_une_table():
    assert not is_table(ex.ProductCreate), (
        "ProductCreate ne doit PAS être une table (pas de table=True)."
    )


def test_exercice4_productcreate_instanciation_valide():
    pc = ex.ProductCreate(name="Clavier mécanique", price=49.90)
    assert pc.price == 49.90


def test_exercice4_productcreate_prix_negatif_refuse():
    assert hasattr(ex, "ProductCreate"), "La classe ProductCreate est introuvable."
    with pytest.raises(Exception):
        ex.ProductCreate(name="Clavier cassé", price=-10)
    # Si ce test échoue : le champ price de ProductCreate n'a pas Field(gt=0).

# --- Lancement des tests ----------------------------------------------------
import types

ex = types.SimpleNamespace(**globals())
_ok = 0
_tests = [(nom, f) for nom, f in list(globals().items()) if nom.startswith("test_")]
for _nom, _test in _tests:
    try:
        _test()
        print("✅", _nom)
        _ok += 1
    except Exception as erreur:
        print("❌", _nom, "→", str(erreur) or type(erreur).__name__)
print(f"\n{_ok}/{len(_tests)} tests réussis" + (" — bravo !" if _ok == len(_tests) else ""))
